# Assistant de recherche sur les projets européens
### Poser une question en français, obtenir une réponse sourcée

Interroge en langage naturel les **projets financés en gestion directe**
(Horizon 2020, Horizon Europe…) à participation française.

> **« Apprendre les textes » ne signifie pas entraîner un modèle.**
> Réentraîner un LLM sur ce corpus coûterait cher, prendrait des jours, et
> produirait un modèle inventant des montants invérifiables. L'approche retenue
> — dite **RAG** — est l'inverse : on *retrouve* les projets pertinents, puis on
> demande au modèle de rédiger **à partir de ces seuls textes**, en citant ses
> sources. Les données restent à jour, chaque affirmation est vérifiable, et
> aucun réentraînement n'est nécessaire.

## Deux modes de fonctionnement

| Mode | Ce qu'il produit | Prérequis |
|---|---|---|
| **Factuel** (par défaut) | projets classés + synthèse chiffrée automatique | aucun |
| **Rédigé** | réponse en langage naturel, avec identifiants cités | accès à un LLM |

Le mode factuel n'invente rien : il compte, additionne et classe. Il répond
déjà à « quels projets », « combien », « quels bénéficiaires ». Le mode rédigé
ajoute l'explication en prose.


## 0 · Paramètres

In [19]:
# Installation si Internet est activé ; sinon le notebook bascule tout seul
# sur scikit-learn (préinstallé sur Kaggle) — aucune action requise.
!pip -q install rank_bm25 ipywidgets 2>/dev/null || echo "(hors ligne : repli automatique)"

import pandas as pd, numpy as np, re, unicodedata, glob, os, json, textwrap

# ── Accès à un modèle de langage ───────────────────────────────────────────
# Le code est volontairement INDÉPENDANT DU FOURNISSEUR : pour en changer,
# seules FOURNISSEUR et LLM_MODELE sont à modifier. Utile si le SGAE reprend
# le projet et souhaite basculer sur Albert (modèle de l'État) ou un autre.
LLM_ACTIF    = True
FOURNISSEUR  = "albert"        # "claude" | "mistral" | "groq" | "openai" | "ollama"
LLM_MODELE   = "gpt-oss-120b"          # ajusté automatiquement via /v1/models

CONFIG_LLM = {
  "albert":  ("https://albert.api.etalab.gouv.fr/v1/chat/completions", "openai"),
  "claude":  ("https://api.anthropic.com/v1/messages",                "anthropic"),
  "mistral": ("https://api.mistral.ai/v1/chat/completions",           "openai"),
  "groq":    ("https://api.groq.com/openai/v1/chat/completions",      "openai"),
  "openai":  ("https://api.openai.com/v1/chat/completions",           "openai"),
  "ollama":  ("http://localhost:11434/v1/chat/completions",           "openai"),
}
LLM_URL, LLM_PROTOCOLE = CONFIG_LLM[FOURNISSEUR]

# ── Clé d'accès — JAMAIS en clair dans le notebook ─────────────────────────
# Kaggle  : Add-ons → Secrets → ajouter « ANTHROPIC_API_KEY »
# Colab   : icône clé à gauche → ajouter le même nom
# Local   : variable d'environnement
import os
# ⚠️ Ne JAMAIS écrire la clé ici. Utilisez les secrets Kaggle
#    (panneau de droite → Secrets → ALBERT_API_KEY), puis redémarrez la session.
# os.environ["ALBERT_API_KEY"] = "…"   # dépannage uniquement, à effacer avant partage
NOM_SECRET = ("ALBERT_API_KEY" if FOURNISSEUR == "albert" else "ANTHROPIC_API_KEY")

# ── Albert API — plateforme d'inférence interministérielle (DINUM) ─────────
# Accès gratuit aux agents de la fonction publique d'État.
# Clé : https://albert.api.etalab.gouv.fr   ·   Doc : guides.ia.numerique.gouv.fr
ALBERT_BASE = "https://albert.api.etalab.gouv.fr/v1"

# ── Modèles Albert — identifiants tels que renvoyés par /v1/models ────────
#   ⚠️ L'endpoint /v1/chat/completions n'accepte QUE le type « text-generation ».
#      Les modèles classés « image-text-to-text » y sont refusés (erreur 422).
#
#   text-generation            gpt-oss-120b                     ← retenu, tâches complexes
#                              qwen3-coder-30b-a3b-instruct     spécialisé code
#                              deepseek-v4-flash-0731           code agentique
#   image-text-to-text         mistral-small-3-2-24b-instruct-2506, ministral-3-8b…
#   text-embeddings-inference  bge-m3                           ← vectorisation
#   text-classification        bge-reranker-v2-m3               ← reclassement
ALBERT_EMB_MODELE    = "bge-m3"
ALBERT_RERANK_MODELE = "bge-reranker-v2-m3"

# Sélection automatique à partir de /v1/models : garantit des identifiants
# valides même si le catalogue évolue.
AUTO_MODELES = True

# Quotas en offre « Expérimentation » (gratuite, tout agent de l'État) :
#   bge-m3 : 500 requêtes/min, 50 000/jour → 77 000 projets par lots de 64
#            représentent environ 1 200 requêtes, soit 2,5 min au débit maximal.
ALBERT_RPM = 450          # marge sous le plafond de 500

def _lire_cle(nom=NOM_SECRET):
    try:                                        # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nom)
    except Exception: pass
    try:                                        # Colab
        from google.colab import userdata
        return userdata.get(nom)
    except Exception: pass
    return os.environ.get(nom, "")              # local

LLM_CLE = (_lire_cle() or "").strip().strip('"').strip("'")
if LLM_ACTIF and not LLM_CLE:
    print(f"⚠️ Secret « {NOM_SECRET} » introuvable — bascule en mode factuel.\n"
          "   Kaggle : Add-ons → Secrets.  Colab : icône clé.  Local : variable d'environnement.")
    LLM_ACTIF = False
elif LLM_ACTIF:
    print(f"✅ Modèle configuré : {FOURNISSEUR} / {LLM_MODELE}")

K_CONTEXTE = 400      # nombre maximal de projets retenus par requête
K_AFFICHES = None     # None = afficher TOUS les projets retenus

# ══ PRINCIPE : permissif à la récupération, sévère au reclassement ════════
# Un projet qui parle du sujet avec d'autres mots doit pouvoir être récupéré :
# c'est le reclassement (modèle Albert) qui fait ensuite le tri fin.
# Filtrer trop tôt rend ces projets définitivement invisibles.
#
# ── Filtres de présélection — DÉSACTIVÉS par défaut ───────────────────────
# COUVERTURE_MIN : part des termes de la question qu'un projet doit contenir.
#   0.5 = au moins la moitié. C'est le filtre le plus efficace contre le bruit :
#   un projet ne partageant qu'un terme banal est écarté.
#   Abaisser à 0.34 pour élargir, monter à 0.7 pour resserrer.
COUVERTURE_MIN = 0

# SCORE_MIN_RELATIF : score minimal en % de celui du meilleur résultat.
#   Écarte la longue traîne des projets faiblement liés.
SCORE_MIN_RELATIF = 0


✅ Modèle configuré : albert / gpt-oss-120b


## 0bis · Vérification de l'accès Albert

Contrôle l'état de l'API, la validité de la clé, les quotas restants et la liste
réelle des modèles — classés par type. À exécuter en premier en cas de doute :
le message d'erreur renvoyé par Albert est explicite (clé invalide, compte
expiré, budget épuisé, quota dépassé).

In [20]:
import requests
print(f"Clé lue : {len(LLM_CLE)} caractères, début « {LLM_CLE[:8]}… »")
print("Espaces parasites :", LLM_CLE != LLM_CLE.strip())

r = requests.get("https://albert.api.etalab.gouv.fr/v1/me",
                 headers={"Authorization": f"Bearer {LLM_CLE.strip()}"}, timeout=20)
print(r.status_code, r.text[:200])

Clé lue : 158 caractères, début « sk-eyJhb… »
Espaces parasites : False
200 {"object":"userInfo","id":18418,"email":"modou.wade@sgae.gouv.fr","name":"MODOU WADE","organization_id":80,"budget":null,"permissions":[],"limits":[{"router_id":342,"type":"rpm","value":500},{"router_


In [21]:
def diagnostic_albert():
    import requests
    H = {"Authorization": f"Bearer {LLM_CLE}"}

    # 1. l'API répond-elle ?
    try:
        r = requests.get("https://albert.api.etalab.gouv.fr/health", timeout=20)
        print(f"API        : {'disponible' if r.ok else 'indisponible'} (HTTP {r.status_code})")
    except Exception as ex:
        print(f"API        : injoignable ({type(ex).__name__})")
        print("             → vérifiez qu'Internet est activé (Settings → Internet)")
        return

    if not LLM_CLE:
        print(f"Clé        : {len(LLM_CLE)} caractères, début « {LLM_CLE[:6]}… »")
        return

    # 2. la clé est-elle valide ?
    try:
        r = requests.get(f"{ALBERT_BASE}/me", headers=H, timeout=20)
        if r.status_code == 401:
            print("Clé        : INVALIDE — le serveur refuse ce jeton.")
            print("             Causes fréquentes : clé révoquée, clé remplacée sans")
            print("             redémarrage de la session, ou copie incomplète.")
            print("             → créez une clé sur le playground Albert, mettez à jour")
            print("               le secret, puis REDÉMARREZ la session (Run → Restart).")
            return
        if r.status_code == 403:
            print("Compte     : EXPIRÉ — contactez albert.api@numerique.gouv.fr"); return
        r.raise_for_status()
        me = r.json()
        print(f"Compte     : {me.get('email', '?')}")
        print(f"Budget     : {me.get('budget', '—')}")
        for lim in (me.get("limits") or [])[:6]:
            print(f"   quota   : {lim}")
    except Exception as ex:
        print(f"Clé        : vérification impossible ({type(ex).__name__})")

    # 3. quels modèles, de quel type ?
    try:
        r = requests.get(f"{ALBERT_BASE}/models", headers=H, timeout=30)
        r.raise_for_status()
        par_type = {}
        for m in r.json().get("data", []):
            par_type.setdefault(m.get("type", "?"), []).append(m.get("id"))
        print("\nModèles disponibles :")
        for t, ids in par_type.items():
            print(f"   {t}")
            for i in ids: print(f"      {i}")
    except Exception as ex:
        print(f"Modèles    : liste indisponible ({type(ex).__name__})")

diagnostic_albert()


# ── Sélection automatique des modèles, par type déclaré ───────────────────
def choisir_modeles(verbeux=True):
    """Interroge /v1/models et retient un modèle valide pour chaque usage.

    Évite l'erreur 422 « Model has wrong type » : chaque endpoint n'accepte
    qu'un type précis.
    """
    global LLM_MODELE, ALBERT_EMB_MODELE, ALBERT_RERANK_MODELE
    if not (AUTO_MODELES and LLM_CLE):
        return
    import requests
    try:
        r = requests.get(f"{ALBERT_BASE}/models",
                         headers={"Authorization": f"Bearer {LLM_CLE}"}, timeout=30)
        r.raise_for_status()
        par_type = {}
        for m in r.json().get("data", []):
            par_type.setdefault(m.get("type", "?"), []).append(m.get("id"))
    except Exception as ex:
        if verbeux: print(f"Modèles : catalogue indisponible ({type(ex).__name__})")
        return

    def retenir(types, preferes, actuel):
        dispo = [i for t in types for i in par_type.get(t, [])]
        if actuel in dispo:
            return actuel
        for p in preferes:
            for i in dispo:
                if p in i.lower(): return i
        return dispo[0] if dispo else actuel

    LLM_MODELE = retenir(["text-generation"],
                         ["gpt-oss", "mistral", "ministral"], LLM_MODELE)
    ALBERT_EMB_MODELE = retenir(["text-embeddings-inference"],
                                ["bge-m3", "embedding"], ALBERT_EMB_MODELE)
    ALBERT_RERANK_MODELE = retenir(["text-classification"],
                                   ["rerank"], ALBERT_RERANK_MODELE)
    if verbeux:
        print(f"Chat        : {LLM_MODELE}")
        print(f"Vectorisation : {ALBERT_EMB_MODELE}")
        print(f"Reclassement  : {ALBERT_RERANK_MODELE}")

choisir_modeles()


API        : disponible (HTTP 200)
Compte     : modou.wade@sgae.gouv.fr
Budget     : None
   quota   : {'router_id': 342, 'type': 'rpm', 'value': 500}
   quota   : {'router_id': 342, 'type': 'rpd', 'value': 50000}
   quota   : {'router_id': 342, 'type': 'tpm', 'value': None}
   quota   : {'router_id': 342, 'type': 'tpd', 'value': None}
   quota   : {'router_id': 343, 'type': 'rpm', 'value': 500}
   quota   : {'router_id': 343, 'type': 'rpd', 'value': 50000}

Modèles disponibles :
   text-generation
      gpt-oss-120b
      qwen3-coder-30b-a3b-instruct
      deepseek-v4-flash-0731
   image-text-to-text
      ministral-3-8b-instruct-2512
      mistral-small-3-2-24b-instruct-2506
      lightonocr-2-1b
      gemma-4-31b-it
   text-embeddings-inference
      bge-m3
      qwen3-vl-embedding-8b
   text-classification
      bge-reranker-v2-m3
   automatic-speech-recognition
      whisper-large-v3
Chat        : gpt-oss-120b
Vectorisation : bge-m3
Reclassement  : bge-reranker-v2-m3


## 1 · Chargement du corpus

Réutilise les trois fichiers de l'API Funding & Tenders. Si vous avez sauvegardé
un index lors d'une session précédente, il est rechargé directement.

In [22]:
index_pret = None
for _b in ("/kaggle/input/*/", "./"):
    _r = glob.glob(_b + "index_recherche.pkl")
    if _r: index_pret = _r[0]; break

if index_pret:
    import pickle
    with open(index_pret, "rb") as f: o = pickle.load(f)
    df, tokens = o["df"], o["tokens"]
    for _alt in ("CONTRIBUTION_PAYS", "CONTRIBUTION"):
        if _alt in df.columns and "CONTRIBUTION_FR" not in df.columns:
            df = df.rename(columns={_alt: "CONTRIBUTION_FR"})
    print(f"Index rechargé : {len(df)} projets")
else:
    LECTURE = dict(sep=";", dtype=str, on_bad_lines="skip", encoding_errors="replace")

    # ── Forçage manuel (laisser None pour la détection automatique) ─────────────
    CHEMIN_PARTICIPANTS = None
    CHEMIN_PROJETS      = None
    CHEMIN_RESUMES      = None

    # ── Inventaire de ce qui est réellement monté ──────────────────────────────
    _tous = [f for f in glob.glob("/kaggle/input/**/*", recursive=True) if os.path.isfile(f)]
    if not _tous:
        _tous = [f for f in glob.glob("./*") if os.path.isfile(f)]

    print(f"{len(_tous)} fichier(s) accessible(s) :")
    for f in _tous:
        print(f"   {f}   ({os.path.getsize(f)/1e6:.1f} Mo)")
    if not _tous:
        raise FileNotFoundError(
            "Aucun fichier. Panneau de droite → + Add Input → onglet « Your Datasets » "
            "→ ajoutez le dataset contenant projects / participants / summaries. "
            "Vérifiez aussi que son traitement par Kaggle est terminé.")

    # ── Identification par le CONTENU (l'en-tête), pas par le nom ──────────────
    SIGNATURES = {
        "participants": ("legalname", "countrycode", "eucontribution"),
        "projets":      ("title", "objective", "eucontributionamount"),
        "resumes":      ("workperformed", "finalresults"),
    }

    def _entete(chemin):
        try:
            with open(chemin, encoding="utf-8", errors="replace") as h:
                return h.readline().lower()
        except Exception:
            return ""

    def _identifier(tous):
        trouve = {}
        for f in tous:
            if not f.lower().endswith((".csv", ".xlsx", ".xls")):
                continue
            tete = _entete(f)
            for role, mots in SIGNATURES.items():
                if role in trouve:
                    continue
                if sum(m in tete for m in mots) >= 2:
                    trouve[role] = f
        return trouve

    _auto = _identifier(_tous)
    F_PART = CHEMIN_PARTICIPANTS or _auto.get("participants")
    F_PROJ = CHEMIN_PROJETS      or _auto.get("projets")
    F_SUMM = CHEMIN_RESUMES      or _auto.get("resumes")

    print("\nIdentification :")
    for lib, f in (("participants", F_PART), ("projets", F_PROJ), ("résumés", F_SUMM)):
        print(f"   {lib:14s} {os.path.basename(f) if f else '— introuvable'}")

    if not (F_PART and F_PROJ):
        raise FileNotFoundError(
            "Les fichiers « participants » et « projects » sont indispensables.\n"
            "Deux causes possibles :\n"
            "  1. Le dataset rattaché n'est pas le bon — ce notebook attend les trois\n"
            "     fichiers CORDIS (projects, participants, summaries), PAS le fichier FTS.\n"
            "  2. Les en-têtes diffèrent : renseignez CHEMIN_PARTICIPANTS et CHEMIN_PROJETS\n"
            "     en haut de cette cellule avec les chemins affichés ci-dessus.")

    # ── Chargement ─────────────────────────────────────────────────────────────
    COLS_P = ["role","legalName","countryCode.abbreviation","postalAddress.city",
              "activityType","eucontribution","projectId"]
    part = pd.concat(
        [c[c["countryCode.abbreviation"].astype(str).str.upper() == "FR"]
         for c in pd.read_csv(F_PART, usecols=lambda c: c in COLS_P,
                              chunksize=100_000, **LECTURE)],
        ignore_index=True)
    part["eucontribution"] = pd.to_numeric(part["eucontribution"], errors="coerce")
    print(f"\n{len(part)} participations françaises")

    def _j(s, n=4):
        v = [str(x) for x in s.dropna().unique() if str(x).strip()]
        return " | ".join(v[:n]) + (f" (+{len(v)-n})" if len(v) > n else "")

    agg = (part.groupby("projectId")
           .agg(BENEFICIAIRES=("legalName", _j), NB_BENEFICIAIRES=("legalName", "nunique"),
                CONTRIBUTION_FR=("eucontribution", "sum"),
                VILLES=("postalAddress.city", _j), TYPES=("activityType", _j))
           .reset_index())

    COLS_J = ["projectId","reference","title","objective","freeKeywords",
              "euContributionAmount","startDate","programAbbreviation","url"]
    proj = pd.read_csv(F_PROJ, usecols=lambda c: c in COLS_J, **LECTURE)
    proj = proj[proj["projectId"].isin(agg["projectId"])]
    proj["ANNEE"] = pd.to_datetime(proj["startDate"], errors="coerce", utc=True).dt.year
    proj["euContributionAmount"] = pd.to_numeric(proj["euContributionAmount"], errors="coerce")
    df = proj.merge(agg, on="projectId", how="inner")

    if F_SUMM:
        COLS_S = ["projectId","summary","workPerformed","finalResults"]
        blocs = [c[c["projectId"].isin(df["projectId"])]
                 for c in pd.read_csv(F_SUMM, usecols=lambda c: c in COLS_S,
                                      chunksize=20_000, **LECTURE)]
        rap = pd.concat(blocs, ignore_index=True) if blocs else pd.DataFrame(columns=COLS_S)
        if len(rap):
            rap = (rap.fillna("").groupby("projectId")
                   .agg(RAPPORT=("summary", lambda s: " ".join(s)[:4000]),
                        TRAVAUX=("workPerformed", lambda s: " ".join(s)[:3000]),
                        RESULTATS=("finalResults", lambda s: " ".join(s)[:2000])).reset_index())
            df = df.merge(rap, on="projectId", how="left")

    for c in ("RAPPORT","TRAVAUX","RESULTATS"):
        if c not in df.columns: df[c] = ""
    df[["RAPPORT","TRAVAUX","RESULTATS"]] = df[["RAPPORT","TRAVAUX","RESULTATS"]].fillna("")
    tokens = None
    print(f"{len(df)} projets français chargés "
          f"({int(df['ANNEE'].min())}–{int(df['ANNEE'].max())})")

    # ── Harmonisation des noms de colonnes ────────────────────────────────────
    for _alt in ("CONTRIBUTION_PAYS", "CONTRIBUTION"):
        if _alt in df.columns and "CONTRIBUTION_FR" not in df.columns:
            df = df.rename(columns={_alt: "CONTRIBUTION_FR"})
    if "CONTRIBUTION_FR" not in df.columns:
        df["CONTRIBUTION_FR"] = np.nan


4 fichier(s) accessible(s) :
   /kaggle/input/datasets/modouwadewade/recherche-thematique/summaries(2).csv   (333.7 Mo)
   /kaggle/input/datasets/modouwadewade/recherche-thematique/projects(3).csv   (184.8 Mo)
   /kaggle/input/datasets/modouwadewade/recherche-thematique/participants(2).csv   (87.0 Mo)
   /kaggle/input/datasets/modouwadewade/kohesio/FR_KOHESIO_14_27.xlsx   (71.1 Mo)

Identification :
   participants   participants(2).csv
   projets        projects(3).csv
   résumés        summaries(2).csv

32424 participations françaises
15694 projets français chargés (2003–2025)


## 1bis · Ajout du corpus Kohesio (gestion partagée)

Le corpus CORDIS couvre la **gestion directe**. Ce bloc y ajoute les projets
**Kohesio** — gestion partagée, FEDER, FSE+, FTJ — soit 61 602 projets français
de 2014 à 2027.

> **Avantage propre à Kohesio** : chaque projet porte son résumé **en français
> et en anglais** sur la même ligne. Une question posée en français y trouve
> directement ses réponses, sans dépendre du moteur sémantique ni du lexique.

Les deux sources sont normalisées vers un **schéma commun**, puis empilées avec
une colonne `CANAL`. La recherche porte alors sur l'ensemble, et chaque résultat
indique de quel canal il provient.

| Champ commun | CORDIS (directe) | Kohesio (partagée) |
|---|---|---|
| identifiant | `projectId` | `Operation_Unique_Identifier` |
| titre | `title` | `Operation_Name_Programme_Language` |
| texte | objectif + rapports | résumés FR **et** EN |
| bénéficiaires | participants français | `Beneficiary_Name` |
| montant | contribution des partenaires FR | `Project_EU_Budget` |
| programme | `programAbbreviation` | `Programme_Name` |

> Les montants ne sont **pas additionnables** entre canaux : contribution
> contractualisée d'un côté, montant programmé de l'autre.

In [23]:
AJOUTER_KOHESIO = True
# Nombre maximal de projets Kohesio chargés. None = tous (61 602).
# L'indexation complète demande environ 6 Go de mémoire : suffisant sur Kaggle,
# insuffisant sur une machine modeste — réduire alors cette valeur.
LIMITE_KOHESIO = None

def _trouver_kohesio():
    """Repère le fichier Kohesio par son CONTENU, quel que soit son nom.

    Un fichier Kohesio se reconnaît à ses colonnes : Operation_Unique_Identifier,
    Beneficiary_Name, Project_EU_Budget. La recherche est récursive dans tous les
    datasets montés.
    """
    candidats = [f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                 if os.path.isfile(f) and f.lower().endswith((".xlsx", ".xls", ".csv"))]
    if not candidats:
        candidats = [f for f in glob.glob("./**/*", recursive=True)
                     if os.path.isfile(f) and f.lower().endswith((".xlsx", ".xls", ".csv"))]

    SIGNES = ("operation_unique_identifier", "beneficiary_name", "project_eu_budget",
              "operation_summary")
    for f in candidats:
        try:
            if f.lower().endswith(".csv"):
                with open(f, encoding="utf-8", errors="replace") as h:
                    tete = h.readline().lower()
            else:
                tete = " ".join(map(str, pd.read_excel(f, nrows=0).columns)).lower()
        except Exception:
            continue
        if sum(s in tete for s in SIGNES) >= 2:
            return f
    # diagnostic si rien n'a été reconnu
    print("   Fichiers examinés :", [os.path.basename(f) for f in candidats] or "aucun")
    return None

def charger_kohesio(chemin):
    lire = (pd.read_excel if chemin.lower().endswith((".xlsx", ".xls"))
            else (lambda p: pd.read_csv(p, sep=";", on_bad_lines="skip",
                                        encoding_errors="replace", dtype=str)))
    k = lire(chemin) if chemin.lower().endswith((".xlsx", ".xls")) else lire(chemin)
    if not chemin.lower().endswith(".csv"):
        k = k.astype(str).replace("nan", "")

    o = pd.DataFrame(index=k.index)
    o["projectId"]     = k.get("Operation_Unique_Identifier", "").astype(str).str.extract(r"(Q\d+)")[0]
    o["title"]         = k.get("Operation_Name_Programme_Language", "")
    # Troncature : les résumés Kohesio atteignent 1 300 caractères ; conservés
    # entiers sur 61 602 projets, l'index dépasse la mémoire disponible.
    # Le début d'un résumé porte l'essentiel du sens.
    LG_FR, LG_EN = 800, 500
    o["objective"]     = (k.get("Operation_Summary_Programme_Language", "").astype(str).str.slice(0, LG_FR)
                          + " " +
                          k.get("Operation_Summary_English", "").astype(str).str.slice(0, LG_EN)).str.strip()
    o["freeKeywords"]  = (k.get("Category_Label", "").astype(str) + " " +
                          k.get("Policy_Objective_Label", "").astype(str)).str.strip()
    o["BENEFICIAIRES"] = k.get("Beneficiary_Name", "")
    o["NB_BENEFICIAIRES"] = 1
    o["CONTRIBUTION_FR"]  = pd.to_numeric(k.get("Project_EU_Budget"), errors="coerce")
    o["euContributionAmount"] = pd.to_numeric(k.get("Total_Eligible_Expenditure_amount"), errors="coerce")
    o["ANNEE"] = pd.to_datetime(k.get("Operation_Start_Date"), dayfirst=True,
                                errors="coerce").dt.year
    o["programAbbreviation"] = k.get("Programme_Name", "")
    o["VILLES"] = k.get("LAU_Labels", "")
    for c in ("RAPPORT", "TRAVAUX", "RESULTATS"):
        o[c] = ""
    o["CANAL"] = "Gestion partagée"
    return o[o["title"].astype(str).str.strip().ne("")]

if AJOUTER_KOHESIO:
    _ch = _trouver_kohesio()
    if _ch:
        if "CANAL" not in df.columns:
            df["CANAL"] = "Gestion directe"
        _koh = charger_kohesio(_ch)
        if LIMITE_KOHESIO:
            _koh = _koh.head(LIMITE_KOHESIO)
            print(f"(limité à {LIMITE_KOHESIO} projets Kohesio)")
        _avant = len(df)
        df = pd.concat([df, _koh], ignore_index=True, sort=False)
        for c in ("RAPPORT", "TRAVAUX", "RESULTATS", "freeKeywords", "objective"):
            if c in df.columns: df[c] = df[c].fillna("")
        print(f"Kohesio ajouté : {len(_koh)} projets  ({os.path.basename(_ch)})")
        print(f"Corpus total : {len(df)} projets "
              f"({_avant} gestion directe + {len(_koh)} gestion partagée)")
        print(df["CANAL"].value_counts().to_string())
    else:
        print("⚠️ Fichier Kohesio introuvable — corpus limité à la gestion directe.")
        _vus = [f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                if os.path.isfile(f)]
        if _vus:
            print("   Fichiers accessibles :")
            for _f in _vus:
                print(f"     {_f}   ({os.path.getsize(_f)/1e6:.1f} Mo)")
            print("   → si le fichier Kohesio figure ci-dessus, renseignez "
                  "CHEMIN_KOHESIO avec son chemin complet.")
        else:
            print("   Aucun fichier monté : rattachez le dataset "
                  "(+ Add Input → Your Datasets).")
        if "CANAL" not in df.columns: df["CANAL"] = "Gestion directe"
else:
    if "CANAL" not in df.columns: df["CANAL"] = "Gestion directe"


Kohesio ajouté : 61587 projets  (FR_KOHESIO_14_27.xlsx)
Corpus total : 77281 projets (15694 gestion directe + 61587 gestion partagée)
CANAL
Gestion partagée    61587
Gestion directe     15694


## 2 · Index lexical (mots exacts)

In [24]:
# ── Moteur lexical : BM25 si disponible, sinon TF-IDF (scikit-learn) ───────
# scikit-learn est préinstallé sur Kaggle : le notebook fonctionne donc même
# sans accès Internet. Les deux moteurs exposent la même méthode get_scores().

STOP = set("""the and for with from this that are was were will been has have its our their
project research development based using into within order also more than which such between
au aux avec ce ces dans de des du elle en et eux il je la le les leur lui mais me meme mes moi
mon ne nos notre nous on ou par pas pour que qui sa se ses son sur une vos votre vous est sont
quel quels quelle quelles combien quoi comment pourquoi""".split())

def normaliser(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().lower()
    return [m for m in re.findall(r"[a-z0-9]{3,}", t) if m not in STOP]

CH = ["title", "objective", "freeKeywords", "RAPPORT", "TRAVAUX", "RESULTATS", "BENEFICIAIRES"]
corpus = df[[c for c in CH if c in df.columns]].fillna("").agg(" ".join, axis=1)

try:
    from rank_bm25 import BM25Okapi
    tokens = [normaliser(t) for t in corpus]
    bm25 = BM25Okapi(tokens)
    MOTEUR = "BM25"
except ImportError:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import linear_kernel

    tokens = [" ".join(normaliser(t)) for t in corpus]
    _vect = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.6)
    _matrice = _vect.fit_transform(tokens)

    class _MoteurTfidf:
        """Interface identique à BM25Okapi : get_scores(liste_de_mots)."""
        def get_scores(self, mots):
            q = _vect.transform([" ".join(mots)])
            return linear_kernel(q, _matrice).ravel()

    bm25 = _MoteurTfidf()
    MOTEUR = "TF-IDF (repli hors ligne)"

# Ensemble des mots de chaque projet : sert au filtre de couverture (précision).
# Les chaînes sont internées pour partager la mémoire entre projets.
import sys as _sys
_interne = _sys.intern
_MOTS_PROJET = [frozenset(_interne(m) for m in (t if isinstance(t, list) else t.split()))
                for t in tokens]

print(f"Index prêt : {len(df)} projets — moteur {MOTEUR}")


Index prêt : 77281 projets — moteur BM25


## 3 · Moteur sémantique multilingue (sens)

Un modèle multilingue projette **le français et l'anglais dans le même espace
de sens**. « Voiture à hydrogène » et « hydrogen vehicle » s'y retrouvent au
même endroit — sans traduction, sans lexique, et **pour n'importe quel sujet**.

Il capte aussi les reformulations qu'aucun lexique ne couvrirait : « pile à
combustible », « mobilité décarbonée », « photovoltaïque », « chimie verte »…

| | Moteur lexical | Moteur sémantique |
|---|---|---|
| Trouve | les mots **exacts**, sigles, noms propres | le **sens**, quelle que soit la formulation |
| Langue | dépend du lexique | français ↔ anglais nativement |
| Rate | les reformulations | parfois les sigles précis |

Les deux sont **fusionnés** : chacun compense la faiblesse de l'autre.

> Prérequis : **GPU** et **Internet** activés dans les Settings de Kaggle
> (téléchargement du modèle, ~500 Mo, une seule fois). Sans eux, le notebook
> reste en mode lexical seul — aucune erreur.

In [25]:
UTILISER_EMBEDDINGS = True
SOURCE_EMBEDDINGS   = "albert"   # "albert" (API DINUM) | "local" (modèle téléchargé) | "aucun"
TAILLE_LOT_EMB      = 64         # textes par appel à l'API
FICHIER_VECTEURS    = "vecteurs_projets.npy"

emb_model = emb_corpus = None
_emb_via_api = False

def _lister_modeles_albert():
    """Affiche les modèles d'embeddings disponibles sur Albert API."""
    import requests
    r = requests.get(f"{ALBERT_BASE}/models",
                     headers={"Authorization": f"Bearer {LLM_CLE}"}, timeout=30)
    r.raise_for_status()
    return [m for m in r.json().get("data", [])
            if "embed" in str(m.get("type", "") + m.get("id", "")).lower()]

def encoder_albert(textes, modele=None, lot=None):
    """Encode une liste de textes via POST /v1/embeddings (compatible OpenAI)."""
    import requests, time
    modele = modele or ALBERT_EMB_MODELE
    lot = lot or TAILLE_LOT_EMB
    vecteurs = []
    _intervalle = 60.0 / max(ALBERT_RPM, 1)      # respect du plafond de requêtes/min
    _dernier = 0.0
    for i in range(0, len(textes), lot):
        _attente = _intervalle - (time.time() - _dernier)
        if _attente > 0: time.sleep(_attente)
        _dernier = time.time()
        bloc = [t if str(t).strip() else "—" for t in textes[i:i+lot]]
        for essai in range(1, 4):
            try:
                r = requests.post(f"{ALBERT_BASE}/embeddings",
                    headers={"Authorization": f"Bearer {LLM_CLE}",
                             "Content-Type": "application/json"},
                    json={"model": modele, "input": bloc}, timeout=120)
                r.raise_for_status()
                vecteurs += [d["embedding"] for d in r.json()["data"]]
                break
            except Exception as ex:
                if essai == 3: raise
                time.sleep(2 * essai)
        if (i // lot) % 20 == 0:
            print(f"   {min(i+lot, len(textes))}/{len(textes)} textes encodés", end="\r")
    v = np.array(vecteurs, dtype="float32")
    n = np.linalg.norm(v, axis=1, keepdims=True); n[n == 0] = 1
    return v / n            # normalisation : le cosinus devient un produit scalaire

if UTILISER_EMBEDDINGS and SOURCE_EMBEDDINGS != "aucun":
    # vecteurs déjà calculés ?
    _pkl = None
    for _b in ("/kaggle/input/*/", "./"):
        _r = glob.glob(_b + FICHIER_VECTEURS)
        if _r: _pkl = _r[0]; break

    _txt = (df["title"].fillna("").astype(str) + ". " +
            df["objective"].fillna("").astype(str).str.slice(0, 900)).tolist()

    if _pkl and len(np.load(_pkl)) == len(df):
        emb_corpus = np.load(_pkl); _emb_via_api = (SOURCE_EMBEDDINGS == "albert")
        print(f"Vecteurs rechargés : {emb_corpus.shape}")
        emb_model = True
    elif SOURCE_EMBEDDINGS == "albert" and LLM_CLE:
        try:
            _dispo = _lister_modeles_albert()
            if _dispo:
                print("Modèles d'embeddings disponibles :",
                      ", ".join(m["id"] for m in _dispo[:5]))
                if ALBERT_EMB_MODELE not in [m["id"] for m in _dispo]:
                    ALBERT_EMB_MODELE = _dispo[0]["id"]
                    print(f"→ modèle retenu : {ALBERT_EMB_MODELE}")
            print(f"Encodage de {len(_txt)} projets via Albert API…")
            emb_corpus = encoder_albert(_txt)
            np.save(FICHIER_VECTEURS, emb_corpus)
            emb_model, _emb_via_api = True, True
            print(f"\n✅ Moteur sémantique actif via Albert API — {emb_corpus.shape}")
        except Exception as ex:
            print(f"⚠️ Albert API indisponible ({type(ex).__name__}: {str(ex)[:90]})")
            emb_model = emb_corpus = None
    elif SOURCE_EMBEDDINGS == "local":
        try:
            from sentence_transformers import SentenceTransformer
            emb_model = SentenceTransformer(MODELE_EMB)
            emb_corpus = emb_model.encode(_txt, batch_size=128,
                                          show_progress_bar=True, normalize_embeddings=True)
            np.save(FICHIER_VECTEURS, emb_corpus)
            print(f"✅ Moteur sémantique local — {emb_corpus.shape}")
        except Exception as ex:
            print(f"⚠️ Modèle local indisponible ({type(ex).__name__}) — mode lexical seul.")
            emb_model = emb_corpus = None
    else:
        print("⚠️ Aucune clé Albert — mode lexical seul.")
else:
    print("Moteur sémantique désactivé.")

def encoder_question(q):
    """Vectorise la question avec le même moteur que le corpus."""
    if _emb_via_api:
        return encoder_albert([q])[0]
    return emb_model.encode([q], normalize_embeddings=True)[0]


Vecteurs rechargés : (77281, 1024)


## 4 · Recherche : fusion des deux moteurs

Le corpus est en anglais ; les questions seront posées en français. Un petit
**lexique de correspondance** traduit les termes thématiques courants. C'est une
convention explicite, à compléter selon vos sujets — plus fiable qu'une
traduction automatique dont on ne maîtrise pas les choix.

In [26]:
# Lexique : COMPLÉMENT au moteur sémantique, plus nécessaire pour l'essentiel.
# Utile surtout pour les sigles et termes très spécifiques du domaine.
LEXIQUE = {
 "hydrogène":"hydrogen", "hydrogene":"hydrogen",
 "rénovation":"renovation retrofit", "renovation":"renovation retrofit",
 "insertion":"inclusion integration", "jeunes":"youth",
 "biodiversité":"biodiversity", "biodiversite":"biodiversity",
}

def enrichir_question(q):
    ql = unicodedata.normalize("NFKD", q.lower()).encode("ascii","ignore").decode()
    ajouts = [v for k, v in LEXIQUE.items()
              if unicodedata.normalize("NFKD", k).encode("ascii","ignore").decode() in ql]
    return q + (" " + " ".join(ajouts) if ajouts else "")

SEUIL_SEMANTIQUE = 0.08   # similarité minimale pour admettre un projet SANS mot commun
                          # (0.25 était trop strict : écartait les reformulations)

def recuperer(question, k=K_CONTEXTE):
    """Fusionne le classement lexical et le classement sémantique.

    Fusion par rangs réciproques : un projet bien classé par l'un OU l'autre
    moteur remonte. Évite de calibrer deux échelles de score non comparables.
    """
    # -- classement lexical --
    s_lex = bm25.get_scores(normaliser(enrichir_question(question)))
    rang_lex = {int(i): r for r, i in enumerate(np.argsort(s_lex)[::-1])}

    # -- classement sémantique --
    rang_sem, s_sem = {}, None
    if emb_model is not None:
        q_vec = encoder_question(question)
        s_sem = emb_corpus @ q_vec
        rang_sem = {int(i): r for r, i in enumerate(np.argsort(s_sem)[::-1])}

    # -- fusion --
    K, n = 60, len(df)
    fusion = {}
    for i in range(n):
        f = 1.0 / (K + rang_lex.get(i, n))
        if rang_sem: f += 1.0 / (K + rang_sem.get(i, n))
        fusion[i] = f

    ordre = sorted(fusion, key=fusion.get, reverse=True)

    # ── Filtre 1 : correspondance minimale ────────────────────────────────
    if emb_model is None:
        ordre = [i for i in ordre if s_lex[i] > 0]
    else:
        ordre = [i for i in ordre if s_lex[i] > 0 or s_sem[i] > SEUIL_SEMANTIQUE]

    # ── Filtre 2 : COUVERTURE DES TERMES (précision) ──────────────────────
    # Un projet doit contenir une part minimale des termes distinctifs de la
    # question. C'est ce qui écarte les projets partageant un mot banal sans
    # traiter le sujet.
    mots_q = set(normaliser(enrichir_question(question)))
    if mots_q and COUVERTURE_MIN > 0:
        requis = max(1, int(np.ceil(len(mots_q) * COUVERTURE_MIN)))
        garde = []
        for i in ordre:
            presents = sum(1 for m in mots_q if m in _MOTS_PROJET[i])
            if presents >= requis:
                garde.append((i, presents))
            elif emb_model is not None and s_sem[i] > 0.45:
                garde.append((i, presents))      # forte proximité de sens : conservé
        ordre = [i for i, _ in garde]
        _couv = {i: p / len(mots_q) for i, p in garde}
    else:
        _couv = {i: 1.0 for i in ordre}

    ordre = ordre[:k]
    r = df.iloc[ordre].copy()
    if len(ordre):
        vals = np.array([fusion[i] for i in ordre])
        r["score"] = (vals / vals.max() * 100).round(1)
        r["couverture"] = [round(_couv.get(i, 0) * 100) for i in ordre]
        r["origine"] = ["sens" if (rang_sem and rang_sem.get(i, n) < rang_lex.get(i, n))
                        else "mots" for i in ordre]
        # ── Filtre 3 : score relatif minimal ──────────────────────────────
        r = r[r["score"] >= SCORE_MIN_RELATIF]
    return r


## 4bis · Recherche multi-thématique (« ou »)

Permet d'interroger **plusieurs sujets en une seule recherche** :
« hydrogène ou photovoltaïque ou éolien ».

> **Pourquoi une fonction distincte.** Le filtre de précision exige qu'un projet
> contienne une part minimale des termes de la question. Sur une requête
> multi-thèmes, ce filtre écarterait les projets ne traitant qu'un seul des
> sujets — exactement l'inverse de l'effet recherché. Chaque thème est donc
> cherché **séparément**, puis les résultats sont fusionnés.

Chaque projet retenu est **étiqueté** du ou des thèmes qui l'ont fait remonter.
Un projet apparaissant sous plusieurs thèmes est conservé une seule fois, avec
la liste de ses thèmes — utile pour repérer les projets à l'intersection de
plusieurs priorités.

In [27]:
SEPARATEURS = r"\s+ou\s+|\s+or\s+|\s*[|;/]\s*|\s*,\s*"

def decouper_themes(question):
    """Découpe « hydrogène ou photovoltaïque » en thèmes distincts."""
    if isinstance(question, (list, tuple)):
        return [str(t).strip() for t in question if str(t).strip()]
    parts = [p.strip() for p in re.split(SEPARATEURS, str(question), flags=re.I)]
    return [p for p in parts if p] or [str(question).strip()]

def rechercher_multi(question, k_par_theme=60, k_total=None):
    """Recherche plusieurs thèmes et fusionne les résultats.

    Chaque thème est cherché séparément — le filtre de couverture s'applique
    donc thème par thème, et non sur la question entière.
    """
    themes = decouper_themes(question)

    # Reclassement : défini plus bas dans le notebook. S'il est actif, il
    # remplace la présélection par une évaluation de pertinence question/projet.
    _chercher = (rechercher_reclasse
                 if ("rechercher_reclasse" in globals() and RERANK_ACTIF and LLM_CLE)
                 else recuperer)

    if len(themes) == 1:
        r = _chercher(themes[0], k=k_par_theme)
        if not r.empty:
            r = r.copy(); r["THEMES"] = themes[0]; r["NB_THEMES"] = 1
        return r, themes

    par_projet = {}          # projectId -> ligne + thèmes + meilleur score
    for th in themes:
        r = _chercher(th, k=k_par_theme)
        for _, x in r.iterrows():
            pid = x["projectId"]
            if pid in par_projet:
                par_projet[pid]["themes"].append(th)
                par_projet[pid]["score"] = max(par_projet[pid]["score"], x["score"])
            else:
                par_projet[pid] = {"ligne": x, "themes": [th], "score": x["score"]}

    if not par_projet:
        return df.iloc[0:0].copy(), themes

    lignes = []
    for pid, o in par_projet.items():
        l = o["ligne"].copy()
        l["score"] = round(o["score"], 1)
        l["THEMES"] = " + ".join(dict.fromkeys(o["themes"]))
        l["NB_THEMES"] = len(set(o["themes"]))
        lignes.append(l)

    res = pd.DataFrame(lignes)
    # les projets couvrant plusieurs thèmes d'abord, puis par score
    res = res.sort_values(["NB_THEMES", "score"], ascending=[False, False])
    if k_total:
        res = res.head(k_total)
    return res.reset_index(drop=True), themes

def synthese_themes(res, themes):
    """Récapitulatif par thème : nombre de projets et montant."""
    lignes = []
    for th in themes:
        s = res[res["THEMES"].str.contains(re.escape(th), case=False, na=False)]
        m = pd.to_numeric(s["CONTRIBUTION_FR"], errors="coerce").sum()
        lignes.append({"Thème": th, "Projets": len(s), "Contribution FR (€)": round(m)})
    croises = int((res["NB_THEMES"] > 1).sum()) if len(res) else 0
    return pd.DataFrame(lignes), croises


## 3ter · Expansion de la question par le modèle

Une question courte — « voiture à hydrogène », trois mots — ne peut pas
retrouver un projet intitulé « pile à combustible pour poids lourds ». Le modèle
reformule donc la question en une dizaine de variantes : synonymes, termes
techniques du domaine, sigles, équivalents anglais.

La recherche porte ensuite sur **l'union de ces variantes**, ce qui élargit
considérablement la récupération. Le reclassement se charge du tri.

> Un seul appel API par question, mis en cache. Les variantes générées sont
> affichées : si elles partent dans une mauvaise direction, c'est visible
> immédiatement.

In [28]:
EXPANSION_ACTIVE = True
NB_VARIANTES     = 10
_CACHE_EXPANSION = {}

def etendre_question(question, n=None):
    """Demande au modèle des reformulations du sujet. Renvoie une liste de termes."""
    n = n or NB_VARIANTES
    if not (EXPANSION_ACTIVE and LLM_ACTIF and LLM_CLE):
        return []
    if question in _CACHE_EXPANSION:
        return _CACHE_EXPANSION[question]
    try:
        rep = appeler_llm([
            {"role": "system", "content":
             "Tu aides à retrouver des projets de recherche européens. À partir d'un "
             "sujet, propose des formulations alternatives permettant de retrouver des "
             "projets traitant du même thème avec un vocabulaire différent.\n\n"
             "Le corpus est BILINGUE : une partie des projets est décrite en français, "
             "l'autre en anglais. Donne donc OBLIGATOIREMENT :\n"
             f"  • {n//2} formulations en FRANÇAIS (synonymes, termes techniques, sigles)\n"
             f"  • {n//2} formulations en ANGLAIS (termes du domaine tels qu'employés "
             "dans les projets européens)\n\n"
             "Une formulation par ligne, sans numérotation ni commentaire. "
             "Chaque ligne fait 2 à 6 mots."},
            {"role": "user", "content": question}])
        variantes = [l.strip(" -•\t") for l in str(rep).split("\n") if l.strip()]
        variantes = [v for v in variantes
                     if 2 <= len(v.split()) <= 8 and v.lower() != question.lower()][:n]
        _CACHE_EXPANSION[question] = variantes
        return variantes
    except Exception as ex:
        print(f"   (expansion indisponible : {type(ex).__name__})")
        return []

def recuperer_etendu(question, k=None, montrer=True):
    """Récupération élargie : question d'origine + variantes générées.

    Chaque formulation est cherchée séparément, puis les résultats sont réunis.
    Un projet trouvé par plusieurs formulations est conservé une seule fois,
    avec son meilleur rang.
    """
    k = k or K_CONTEXTE
    variantes = etendre_question(question)
    if montrer and variantes:
        _fr = [v for v in variantes if any(c in v.lower() for c in "éèêàçùô")
               or len(set(v.lower().split()) & {"de","du","des","la","le","les","à","pour","en"}) > 0]
        print(f"   Formulations explorées ({len(variantes)}) : "
              + " · ".join(variantes[:8])
              + (f" (+{len(variantes)-8})" if len(variantes) > 8 else ""))

    formulations = [question] + variantes
    vus = {}
    for rang_f, f in enumerate(formulations):
        part = max(k // max(len(formulations), 1), 40)
        r = recuperer(f, k=part)
        for pos, (_, x) in enumerate(r.iterrows()):
            pid = x["projectId"]
            # meilleur rang obtenu, toutes formulations confondues
            if pid not in vus or pos < vus[pid][0]:
                vus[pid] = (pos, x, f)

    if not vus:
        return df.iloc[0:0].copy()
    lignes = []
    for pid, (pos, x, f) in sorted(vus.items(), key=lambda o: o[1][0]):
        l = x.copy(); l["trouve_par"] = f
        lignes.append(l)
    res = pd.DataFrame(lignes).head(k).reset_index(drop=True)
    if montrer:
        print(f"   {len(res)} projets présélectionnés sur {len(formulations)} formulations")
    return res


## 4ter · Reclassement des résultats (reranker)

Albert expose `POST /v1/rerank` avec le modèle `BAAI/bge-reranker-v2-m3`, qui
**juge la pertinence réelle de chaque projet face à la question**. C'est la
réponse au problème des projets hors sujet qui remontaient malgré un vocabulaire
partagé.

Le principe diffère des deux moteurs précédents : au lieu de comparer des
représentations calculées à l'avance, le modèle lit ensemble la question et le
texte du projet, puis attribue un score de pertinence. Beaucoup plus précis,
mais applicable seulement à une poignée de candidats — d'où son usage en
**deuxième passe**, sur les résultats déjà présélectionnés.

> Quotas en offre Expérimentation : 500 requêtes par minute, 50 000 par jour.
> Une requête par recherche suffit, les documents étant envoyés par lot.

In [29]:
RERANK_ACTIF = True        # reclassement des résultats par le modèle Albert
RERANK_CANDIDATS = 300     # projets soumis au reclassement (plus large = meilleur rappel)

# ── Classement, SANS élimination ──────────────────────────────────────────
# Le modèle de reclassement renvoie des scores très bas en valeur absolue :
# un seuil fixe y éliminerait la quasi-totalité des résultats. Tous les projets
# sont donc CONSERVÉS et ordonnés ; les niveaux ci-dessous ne servent qu'à
# qualifier l'affichage, en proportion du meilleur résultat de la recherche.
SEUIL_PERTINENCE = 0       # 0 = ne rien écarter (recommandé)
SEUIL_FAIBLE     = 25      # part du meilleur score, en % : sous ce seuil « à vérifier »
SEUIL_SUR        = 60      # au-dessus : « sûr »

REQUETE_BILINGUE = True     # interroger le reranker en français ET en anglais
_CACHE_QUESTION_EN = {}

def question_en_anglais(q):
    """Traduit la question en anglais — un appel, mis en cache.

    Le corpus mêle textes français et anglais. Comparer une question française
    à un texte anglais désavantage ce dernier : disposer des deux versions
    rétablit l'équilibre.
    """
    if not (REQUETE_BILINGUE and LLM_ACTIF and LLM_CLE):
        return ""
    if q in _CACHE_QUESTION_EN:
        return _CACHE_QUESTION_EN[q]
    try:
        rep = str(appeler_llm([
            {"role": "system", "content":
             "Traduis en anglais ce sujet de recherche, en employant les termes "
             "techniques usuels des projets européens. Réponds par la seule "
             "traduction, sans guillemets ni commentaire."},
            {"role": "user", "content": q}])).strip().strip('"')
        _CACHE_QUESTION_EN[q] = rep
        return rep
    except Exception:
        return ""

TAILLE_LOT_RERANK   = 32    # documents par appel — au-delà, l'API peut refuser
LONGUEUR_DOC_RERANK = 500   # caractères par document soumis au reclassement

def reclasser_albert(question, textes, top_n=None, modele=None):
    """POST /v1/rerank — renvoie [(index, score)] trié par pertinence.

    Les documents sont envoyés par lots : un appel unique sur une liste longue
    dépasse le contexte du modèle et provoque une erreur 422 ou 413.
    """
    import requests
    modele = modele or ALBERT_RERANK_MODELE
    docs = [str(t or "—")[:LONGUEUR_DOC_RERANK] or "—" for t in textes]

    # Une passe par langue de requête ; chaque document garde son MEILLEUR score.
    requetes = [question]
    qen = question_en_anglais(question)
    if qen and qen.lower() != question.lower():
        requetes.append(qen)

    meilleurs = {}
    for q in requetes:
        for debut in range(0, len(docs), TAILLE_LOT_RERANK):
            lot = docs[debut:debut + TAILLE_LOT_RERANK]
            r = requests.post(f"{ALBERT_BASE}/rerank",
                headers={"Authorization": f"Bearer {LLM_CLE}",
                         "Content-Type": "application/json"},
                json={"query": q, "documents": lot, "model": modele,
                      "top_n": len(lot)}, timeout=120)
            r.raise_for_status()
            for d in r.json().get("results", []):
                i = d.get("index")
                if i is None: continue
                idx = debut + i
                s = d.get("relevance_score", d.get("score", 0))
                if idx not in meilleurs or s > meilleurs[idx]:
                    meilleurs[idx] = s
    scores = sorted(meilleurs.items(), key=lambda x: x[1], reverse=True)
    return scores[:top_n] if top_n else scores

def rechercher_reclasse(question, k=K_CONTEXTE, candidats=None):
    """Recherche hybride, puis reclassement des meilleurs candidats.

    Première passe : BM25 + sémantique (rapide, large).
    Seconde passe : le reranker lit question et projet ensemble (précis, étroit).
    """
    # Récupération LARGE (expansion + seuils ouverts), puis tri fin par le modèle
    nb_cand = candidats or RERANK_CANDIDATS
    base = (recuperer_etendu(question, k=nb_cand)
            if (EXPANSION_ACTIVE and LLM_ACTIF and LLM_CLE)
            else recuperer(question, k=nb_cand))
    if base.empty or not (RERANK_ACTIF and LLM_CLE):
        return base.head(k)

    textes = [f"{x['title']}. {str(x.get('objective') or '')[:600]}"
              for _, x in base.iterrows()]
    try:
        ordre = reclasser_albert(question, textes, top_n=k)
    except Exception as ex:
        detail = ""
        rep = getattr(ex, "response", None)
        if rep is not None:
            try:    detail = f" HTTP {rep.status_code} — {rep.json().get('detail', rep.text[:160])}"
            except Exception: detail = f" HTTP {rep.status_code} — {str(rep.text)[:160]}"
        print(f"   ⚠️ Reclassement indisponible : {type(ex).__name__}{detail}")
        print("      Causes fréquentes : lot trop volumineux (réduire RERANK_CANDIDATS "
              "ou LONGUEUR_DOC_RERANK), quota atteint (429), modèle occupé (503).")
        return base.head(k)

    idx = [i for i, _ in ordre if i is not None and i < len(base)]
    r = base.iloc[idx].copy()
    sc = [s for _, s in ordre][:len(idx)]
    if sc:
        r["score_brut"] = [round(s, 4) for s in sc]
        # Score ramené en proportion du meilleur résultat : les valeurs absolues
        # du reranker sont trop basses pour être lisibles ou seuillées.
        m = max(sc) or 1
        r["pertinence"] = [round(s / m * 100, 1) for s in sc]
        r["score"] = r["pertinence"]
        r["fiabilite"] = pd.cut(r["pertinence"],
                                bins=[-0.1, SEUIL_FAIBLE, SEUIL_SUR, 100.1],
                                labels=["à vérifier", "probable", "sûr"])
        if SEUIL_PERTINENCE > 0:
            avant = len(r)
            r = r[r["pertinence"] >= SEUIL_PERTINENCE]
            if len(r) < avant:
                print(f"   ({avant - len(r)} projet(s) écarté(s))")
    print(f"   {len(r)} projets classés par le reclassement")
    return r


## 5 · Réponse factuelle (sans LLM)

Produit une **synthèse chiffrée** : nombre de projets, montants, bénéficiaires
récurrents, répartition dans le temps. Rien n'est inventé — tout est calculé.

In [30]:
def repondre_factuel(question, k=None):
    k = k or 8                      # K_AFFICHES peut valoir None (= tout afficher)
    r=recuperer(question, k=max(k, K_CONTEXTE))
    if r.empty:
        return f"Aucun projet ne correspond à « {question} ».\n" \
               "Essayez des termes plus généraux, ou complétez le LEXIQUE."

    contrib=pd.to_numeric(r["CONTRIBUTION_FR"],errors="coerce")
    top=r.head(k)
    lignes=[f"**{len(r)} projets** correspondent à « {question} ».",
            f"Contribution européenne aux partenaires français : "
            f"**{contrib.sum():,.0f} €**".replace(","," "),
            ""]
    if r["ANNEE"].notna().any():
        lignes.append(f"Période : {int(r['ANNEE'].min())} – {int(r['ANNEE'].max())}")

    # bénéficiaires les plus présents
    noms=[n.strip() for s in r["BENEFICIAIRES"].dropna() for n in str(s).split("|")
          if n.strip() and not n.strip().startswith("(+")]
    if noms:
        freq=pd.Series(noms).value_counts().head(5)
        lignes.append("\n**Bénéficiaires les plus présents :**")
        lignes += [f"  • {n} ({c} projet{'s' if c>1 else ''})" for n,c in freq.items()]

    lignes.append("\n**Projets les plus pertinents :**")
    for _,x in top.iterrows():
        m=pd.to_numeric(x["CONTRIBUTION_FR"],errors="coerce")
        m=f"{m:,.0f} €".replace(","," ") if pd.notna(m) else "n.c."
        an=int(x["ANNEE"]) if pd.notna(x["ANNEE"]) else "----"
        lignes.append(f"  [{x['projectId']}] {str(x['title'])[:70]}")
        lignes.append(f"      {an} · {m} · {str(x['BENEFICIAIRES'])[:60]}")
    return "\n".join(lignes)

print(repondre_factuel("voiture à hydrogène"))


**400 projets** correspondent à « voiture à hydrogène ».
Contribution européenne aux partenaires français : **703 407 738 €**

Période : 2014 – 2025

**Bénéficiaires les plus présents :**
  • COMMISSARIAT A L ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES (55 projets)
  • CENTRE NATIONAL DE LA RECHERCHE SCIENTIFIQUE CNRS (35 projets)
  • ENGIE (18 projets)
  • L AIR LIQUIDE SA (12 projets)
  • IFP Energies nouvelles (12 projets)

**Projets les plus pertinents :**
  [Q7356822] RETROFIT HYDROGENE RH2
      2023 · 728 719 € · RH2
  [101119353] Rouen Vallée Hydrogène
      2022 · 2 098 125 € · VALOREM SAS | ROUEN VALLEE HYDROGENE | VALREA
  [Q3698795] 12015 VHYCTOR
      2017 · 754 791 € · MAHYTEC
  [Q6840309] Hy-O-T (IoT Hydrogène)
      2020 · 37 778 € · Université de Lorraine
  [Q6840305] Hy-O-T (IoT Hydrogène)
      2020 · 42 625 € · 45-8 GROUP
  [Q6841852] STACKPACK- Unité de puissance pile à combustible hydrogène type PEM po
      2020 · 2 820 316 € · SYMBIO FRANCE
  [Q3680378] FEDER 

## 6 · Traduction des résumés en français

Le corpus est **en anglais**. Une cascade tente, dans l'ordre : le modèle de
langage s'il est actif, puis un traducteur en ligne (`deep-translator`), puis un
modèle local (MarianMT). Si aucun n'est disponible, le texte anglais est affiché
tel quel avec une mention explicite — **jamais de traduction inventée**.

Les traductions sont **mises en cache** : un même résumé n'est traduit qu'une
fois par session.

In [31]:
TRADUIRE = True          # False = afficher l'anglais d'origine
_CACHE_TRAD = {}
_TRADUCTEUR = None       # détecté au premier appel

def _init_traducteur():
    """Choisit le premier traducteur disponible. Ordre : LLM > en ligne > local."""
    global _TRADUCTEUR
    if _TRADUCTEUR is not None:
        return _TRADUCTEUR
    if LLM_ACTIF and LLM_CLE:
        _TRADUCTEUR = ("llm", None)
        return _TRADUCTEUR
    try:
        !pip -q install deep-translator 2>/dev/null
        from deep_translator import GoogleTranslator
        GoogleTranslator(source="en", target="fr").translate("test")
        _TRADUCTEUR = ("deep", GoogleTranslator(source="en", target="fr"))
        return _TRADUCTEUR
    except Exception:
        pass
    try:
        from transformers import pipeline
        _TRADUCTEUR = ("marian", pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr"))
        return _TRADUCTEUR
    except Exception:
        pass
    _TRADUCTEUR = ("aucun", None)
    return _TRADUCTEUR

def traduire(texte, max_car=900):
    """Traduit en français. Renvoie (texte, traduit_oui_non)."""
    texte = str(texte or "").strip()[:max_car]
    if not texte or not TRADUIRE:
        return texte, False
    if texte in _CACHE_TRAD:
        return _CACHE_TRAD[texte], True

    mode, outil = _init_traducteur()
    try:
        if mode == "llm":
            res = appeler_llm([
                {"role": "system", "content": "Traduis en français, fidèlement et sans commentaire."},
                {"role": "user", "content": texte}])
        elif mode == "deep":
            res = outil.translate(texte)
        elif mode == "marian":
            res = " ".join(o["translation_text"] for o in
                           outil([texte[i:i+450] for i in range(0, len(texte), 450)]))
        else:
            return texte, False
        _CACHE_TRAD[texte] = res
        return res, True
    except Exception:
        return texte, False

_mode, _ = _init_traducteur()
_libelles = {"llm": "modèle de langage", "deep": "traducteur en ligne",
             "marian": "modèle local MarianMT", "aucun": "aucun — textes en anglais"}
print(f"Traduction : {_libelles[_mode]}")


# ── Traduction par LOTS : un seul appel pour tous les résumés affichés ─────
# Traduire projet par projet ferait un appel API par fiche. Un lot unique
# divise la latence par le nombre de projets et consomme moins de quota.
def traduire_lot(textes, max_car=900):
    """Traduit plusieurs textes en un appel. Renvoie la liste des traductions."""
    textes = [str(t or "").strip()[:max_car] for t in textes]
    sortie = [None] * len(textes)
    a_faire = []
    for i, t in enumerate(textes):
        if not t or not TRADUIRE:
            sortie[i] = t
        elif t in _CACHE_TRAD:
            sortie[i] = _CACHE_TRAD[t]
        else:
            a_faire.append(i)
    if not a_faire:
        return sortie

    mode, outil = _init_traducteur()
    if mode == "aucun":
        for i in a_faire: sortie[i] = textes[i]
        return sortie

    try:
        if mode == "llm":
            # un seul appel, textes numérotés, réponse attendue au même format
            bloc = "\n\n".join(f"<<{n}>> {textes[i]}" for n, i in enumerate(a_faire, 1))
            rep = appeler_llm([
                {"role": "system", "content":
                 "Traduis en français chaque segment numéroté. Restitue EXACTEMENT "
                 "le même format : <<1>> traduction, <<2>> traduction, etc. "
                 "Aucun commentaire, aucune reformulation, aucun segment omis."},
                {"role": "user", "content": bloc}])
            morceaux = re.split(r"<<\s*(\d+)\s*>>", rep)
            trad = {}
            for j in range(1, len(morceaux) - 1, 2):
                try: trad[int(morceaux[j])] = morceaux[j + 1].strip()
                except ValueError: pass
            for n, i in enumerate(a_faire, 1):
                t = trad.get(n)
                sortie[i] = t or textes[i]
                if t: _CACHE_TRAD[textes[i]] = t
        else:
            for i in a_faire:
                sortie[i], _ = traduire(textes[i], max_car)
    except Exception:
        for i in a_faire:
            if sortie[i] is None: sortie[i] = textes[i]
    return sortie


Traduction : modèle de langage


## 4quater · Lecture approfondie par le modèle

Dernière étape de l'entonnoir. Les étapes précédentes **classent** les projets
par proximité de vocabulaire ou de sens ; celle-ci les fait **lire**.

Pour chaque finaliste, le modèle reçoit l'intégralité du texte disponible —
intitulé, objectif, rapport d'exécution, travaux réalisés, résultats — et
produit trois choses :

| Sortie | Contenu |
|---|---|
| `pertinence_ia` | note de 0 à 100, fondée sur le contenu réel du projet |
| `raison` | en quoi le projet répond (ou non) à la question |
| `resume_ia` | résumé rédigé en français, orienté par la question posée |

C'est ce qui permet de retenir un projet qui traite du sujet **sans en employer
les mots** : le modèle comprend que « pile à combustible embarquée » relève de
la voiture à hydrogène, là où aucun appariement de termes ne l'aurait établi.

> **Pourquoi seulement sur les finalistes.** Faire lire 77 000 projets à chaque
> question est hors de portée en temps comme en quota. L'entonnoir reste donc :
> récupération large → reclassement → lecture approfondie des derniers candidats.
> Augmenter `N_LECTURE` élargit cette dernière passe, au prix de la durée.

In [32]:
LECTURE_IA   = True      # lecture approfondie des finalistes par le modèle
N_LECTURE    = 12        # projets soumis à la lecture (au-delà : plus lent)
LOT_LECTURE  = 4         # projets par appel — au-delà, la qualité se dégrade
SEUIL_IA     = 0         # 0 = ne rien écarter ; la note sert au CLASSEMENT seul
                         # (mettre 25 ou plus pour filtrer, au risque de perdre
                         #  des projets pertinents mal notés)
LONGUEUR_TEXTE_IA = 2500 # caractères de texte transmis par projet

_CACHE_LECTURE = {}

def _texte_complet(x):
    """Rassemble tout le contenu disponible sur un projet."""
    parts = [str(x.get(c) or "") for c in
             ("title", "objective", "RAPPORT", "TRAVAUX", "RESULTATS")]
    return " ".join(p for p in parts if p.strip())[:LONGUEUR_TEXTE_IA]

def analyser_projets(question, lignes):
    """Fait lire un lot de projets au modèle. Renvoie {projectId: {...}}."""
    import json as _json
    bloc = []
    for x in lignes:
        bloc.append(f"### PROJET {x['projectId']}\n"
                    f"Bénéficiaires : {str(x.get('BENEFICIAIRES') or '—')[:150]}\n"
                    f"{_texte_complet(x)}")
    corps = "\n\n".join(bloc)

    consigne = (
      "Tu analyses des projets financés par l'Union européenne pour répondre à une "
      "question thématique.\n\n"
      "Pour CHAQUE projet fourni, produis :\n"
      "  • pertinence : note de 0 à 100 mesurant si le projet traite RÉELLEMENT du "
      "sujet de la question. Un projet qui aborde le thème avec un vocabulaire "
      "différent doit recevoir une note ÉLEVÉE. Un projet qui partage seulement "
      "des mots sans traiter le sujet doit recevoir une note BASSE.\n"
      "  • raison : une phrase expliquant le lien avec la question, ou son absence.\n"
      "  • resume : 3 à 5 phrases en français décrivant ce que fait le projet, "
      "ce qu'il apporte et pour qui. Mets en avant ce qui éclaire la question. "
      "N'invente aucun chiffre ni aucun fait absent du texte.\n\n"
      "Réponds UNIQUEMENT par un objet JSON de la forme :\n"
      '{"projets":[{"id":"...","pertinence":0,"raison":"...","resume":"..."}]}'
    )
    rep = appeler_llm([
        {"role": "system", "content": consigne},
        {"role": "user", "content": f"QUESTION : {question}\n\nPROJETS :\n{corps}"}])

    txt = str(rep).strip()
    deb, fin = txt.find("{"), txt.rfind("}")
    if deb < 0 or fin <= deb:
        raise ValueError("réponse non interprétable")
    donnees = _json.loads(txt[deb:fin + 1])
    return {str(p.get("id")): p for p in donnees.get("projets", [])}

def lire_finalistes(question, res, n=None):
    """Applique la lecture approfondie et réordonne les résultats."""
    if not (LECTURE_IA and LLM_ACTIF and LLM_CLE) or res.empty:
        return res
    n = n or N_LECTURE
    sous = res.head(n).copy()
    lignes = [x for _, x in sous.iterrows()]

    analyses = {}
    for debut in range(0, len(lignes), LOT_LECTURE):
        lot = lignes[debut:debut + LOT_LECTURE]
        cle = (question, tuple(x["projectId"] for x in lot))
        if cle in _CACHE_LECTURE:
            analyses.update(_CACHE_LECTURE[cle]); continue
        try:
            a = analyser_projets(question, lot)
            _CACHE_LECTURE[cle] = a
            analyses.update(a)
        except Exception as ex:
            print(f"   ⚠️ Lecture approfondie interrompue : {type(ex).__name__} — "
                  f"{str(ex)[:90]}")
            break

    if not analyses:
        return res

    sous["pertinence_ia"] = [analyses.get(str(p), {}).get("pertinence")
                             for p in sous["projectId"]]
    sous["raison"]    = [analyses.get(str(p), {}).get("raison", "") for p in sous["projectId"]]
    sous["resume_ia"] = [analyses.get(str(p), {}).get("resume", "") for p in sous["projectId"]]

    avant = len(sous)
    if SEUIL_IA > 0:
        garde = sous["pertinence_ia"].isna() | (sous["pertinence_ia"] >= SEUIL_IA)
        sous = sous[garde]
        if len(sous) < avant:
            print(f"   {avant - len(sous)} projet(s) écarté(s) après lecture")
    _lus = sous["pertinence_ia"].notna().sum()
    print(f"   {_lus} projet(s) analysés en profondeur et reclassés")

    sous = sous.sort_values("pertinence_ia", ascending=False, na_position="last")
    sous["score"] = sous["pertinence_ia"].fillna(sous.get("score", 0))

    # les projets non soumis à la lecture restent à la suite
    reste = res[~res["projectId"].isin(sous["projectId"])]
    return pd.concat([sous, reste], ignore_index=True)


## 7 · Affichage en fiches

Chaque projet est présenté sous forme de **fiche** : titre, montant perçu par les
partenaires français, année, programme, bénéficiaires, et résumé traduit. Une
barre de pertinence situe le projet par rapport aux autres résultats.

In [33]:
from IPython.display import HTML, display

_CSS = """
<style>
.rr-entete{background:#1F3864;color:#fff;padding:14px 18px;border-radius:8px 8px 0 0;
  font-family:system-ui,-apple-system,Segoe UI,sans-serif}
.rr-entete h3{margin:0 0 6px 0;font-size:17px;font-weight:600}
.rr-chiffres{display:flex;gap:26px;flex-wrap:wrap;font-size:13px;opacity:.93}
.rr-chiffres b{font-size:16px;display:block;font-weight:600}
.rr-corps{border:1px solid #d7dee8;border-top:none;border-radius:0 0 8px 8px;
  padding:6px 0 2px 0;background:#fbfcfe;font-family:system-ui,-apple-system,sans-serif}
.rr-fiche{border-left:4px solid #6C8EBF;margin:10px 14px;padding:11px 15px;
  background:#fff;border-radius:0 6px 6px 0;box-shadow:0 1px 3px rgba(31,56,100,.08)}
.rr-titre{font-weight:600;color:#1F3864;font-size:14px;margin-bottom:7px;line-height:1.35}
.rr-badges{display:flex;gap:7px;flex-wrap:wrap;margin-bottom:8px}
.rr-b{font-size:11px;padding:3px 9px;border-radius:11px;background:#eef2f8;color:#34495e}
.rr-b.eur{background:#e4f2e4;color:#2d6a2d;font-weight:600}
.rr-b.an{background:#fdf2dd;color:#8a6116}
.rr-benef{font-size:12px;color:#4a5568;margin-bottom:7px}
.rr-benef span{color:#8a94a6}
.rr-resume{font-size:12.5px;color:#3d4756;line-height:1.55;border-top:1px dashed #e2e8f0;
  padding-top:8px;margin-top:3px}
.rr-vo{font-size:10.5px;color:#a0aab8;font-style:italic;margin-top:5px}
.rr-jauge{height:3px;background:#eef2f8;border-radius:2px;margin-top:9px}
.rr-jauge div{height:100%;background:linear-gradient(90deg,#6C8EBF,#1F3864);border-radius:2px}
.rr-pied{font-size:11px;color:#8a94a6;padding:8px 18px 12px;font-style:italic}
.rr-recap{display:flex;gap:10px;flex-wrap:wrap;margin-top:12px}
.rr-rtheme{font-size:11.5px;background:rgba(255,255,255,.14);padding:5px 11px;
  border-radius:5px}
.rr-rtheme b{display:inline;font-size:13px;margin:0 4px}
.rr-groupe{font-size:12px;font-weight:600;color:#1F3864;text-transform:uppercase;
  letter-spacing:.4px;margin:16px 14px 2px;padding-bottom:5px;
  border-bottom:2px solid #dbe3ee}
.rr-groupe span{font-weight:400;color:#8a94a6;text-transform:none;letter-spacing:0}
.rr-b.prog{background:#e8e4f2;color:#4c3d7a;font-weight:600}
.rr-b.croise{background:#fde8e8;color:#9c2b24}
.rr-b.canal{background:#e2eef6;color:#1c5470}
.rr-b.fiab{font-weight:600}
.rr-b.fiab.sur{background:#dff0da;color:#2d6a2d}
.rr-b.fiab.prob{background:#fdf2dd;color:#8a6116}
.rr-b.fiab.verif{background:#f3f4f6;color:#6b7280}
.rr-raison{font-size:12px;color:#1F3864;font-weight:600;margin:4px 0 6px;
  padding-left:2px}
.rr-resume.ia{border-left:3px solid #6C8EBF;padding-left:10px;background:#fbfcfe}
</style>"""

def _euros(v):
    v = pd.to_numeric(v, errors="coerce")
    if pd.isna(v): return "n.c."
    if v >= 1e6:  return f"{v/1e6:.1f} M€".replace(".", ",")
    if v >= 1e3:  return f"{v/1e3:.0f} k€"
    return f"{v:.0f} €"

def _echapper(t):
    return (str(t or "").replace("&","&amp;").replace("<","&lt;").replace(">","&gt;"))

def _txt(v):
    """Valeur texte sûre : les cellules vides de pandas sont des float (NaN)."""
    return "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v).strip()

def afficher_resultats(question, k=None, resume=True, par_theme=True):
    """Affiche les projets en fiches. Gère les recherches multi-thématiques.

    k = None  → affiche TOUS les projets retenus.
    """
    res, themes = rechercher_multi(question, k_par_theme=K_CONTEXTE, k_total=k)
    _n_avant = len(res)
    if LECTURE_IA and LLM_ACTIF and LLM_CLE and not res.empty:
        res = lire_finalistes(question, res)
    if len(res) != _n_avant:
        print(f"   {_n_avant} projets retenus → {len(res)} après analyse")
    if res.empty:
        display(HTML(_CSS + "<div class='rr-entete'><h3>Aucun résultat</h3>"
            f"<div class='rr-chiffres'>Aucun projet ne correspond à "
            f"« {_echapper(question)} ». Essayez des termes plus généraux, ou "
            f"abaissez COUVERTURE_MIN.</div></div>"))
        return res

    contrib = pd.to_numeric(res["CONTRIBUTION_FR"], errors="coerce")
    annees = res["ANNEE"].dropna()
    periode = f"{int(annees.min())} – {int(annees.max())}" if len(annees) else "—"
    recap, croises = synthese_themes(res, themes)

    h = [_CSS, "<div class='rr-entete'>",
         f"<h3>🔎 {_echapper(' · '.join(themes))}</h3>",
         "<div class='rr-chiffres'>",
         f"<div><b>{len(res)}</b>projets</div>",
         f"<div><b>{_euros(contrib.sum())}</b>contribution UE (France)</div>",
         f"<div><b>{periode}</b>période</div>"]
    if "CANAL" in res.columns and res["CANAL"].nunique() > 1:
        _rc = res["CANAL"].value_counts()
        h.append("<div><b>" + " / ".join(str(v) for v in _rc.values) + "</b>"
                 + " / ".join(_rc.index) + "</div>")
    if len(themes) > 1:
        h.append(f"<div><b>{croises}</b>à l'intersection de plusieurs thèmes</div>")
    h.append("</div>")

    # récapitulatif par thème
    if len(themes) > 1:
        h.append("<div class='rr-recap'>")
        for _, t in recap.iterrows():
            h.append(f"<span class='rr-rtheme'>{_echapper(t['Thème'])}"
                     f"<b>{t['Projets']}</b> projets · "
                     f"{_euros(t['Contribution FR (€)'])}</span>")
        h.append("</div>")
    h.append("</div><div class='rr-corps'>")

    # ── Traduction en UN SEUL appel pour toutes les fiches affichées ──────
    _trads = {}
    if resume:
        _src = {}
        for _, x in res.iterrows():
            if _txt(x.get("resume_ia")):
                continue                      # résumé déjà rédigé par le modèle
            t = x.get("objective") or x.get("RAPPORT") or ""
            if str(t).strip(): _src[x["projectId"]] = str(t)
        if _src:
            try:
                _res_trad = traduire_lot(list(_src.values()))
                _trads = dict(zip(_src.keys(), _res_trad))
            except Exception:
                _trads = {k: v for k, v in _src.items()}

    smax = res["score"].max() if "score" in res else 100
    groupes = ([(t, res[res["THEMES"].str.contains(re.escape(t), case=False, na=False)])
                for t in themes] if (par_theme and len(themes) > 1)
               else [(None, res)])

    deja = set()
    for titre_g, sous in groupes:
        if titre_g:
            h.append(f"<div class='rr-groupe'>{_echapper(titre_g)} "
                     f"<span>({len(sous)} projets)</span></div>")
        for _, x in sous.iterrows():
            cle = (titre_g, x["projectId"])
            if cle in deja: continue
            deja.add(cle)

            prog = _echapper(x.get("programAbbreviation") or "—")
            an = int(x["ANNEE"]) if pd.notna(x.get("ANNEE")) else "—"
            largeur = (x.get("score", 0) / smax * 100) if smax else 0

            bloc = [f"<div class='rr-fiche'>",
                    f"<div class='rr-titre'>{_echapper(x.get('title'))}</div>",
                    "<div class='rr-badges'>",
                    f"<span class='rr-b eur'>{_euros(x.get('CONTRIBUTION_FR'))}</span>",
                    f"<span class='rr-b an'>{an}</span>",
                    f"<span class='rr-b prog'>{prog}</span>",
                    f"<span class='rr-b canal'>{_echapper(x.get('CANAL') or '—')}</span>",
                    f"<span class='rr-b'>#{_echapper(x.get('projectId'))}</span>"]
            if x.get("NB_THEMES", 1) > 1:
                bloc.append(f"<span class='rr-b croise'>{_echapper(x['THEMES'])}</span>")
            if pd.notna(x.get("pertinence_ia")):
                _n = float(x["pertinence_ia"])
                _cl = "sur" if _n >= 70 else ("prob" if _n >= 45 else "verif")
                bloc.append(f"<span class='rr-b fiab {_cl}'>analyse IA "
                            f"{_n:.0f}/100</span>")
            elif pd.notna(x.get("pertinence")):
                _f = str(x.get("fiabilite") or "")
                _cl = {"sûr": "sur", "probable": "prob", "à vérifier": "verif"}.get(_f, "")
                bloc.append(f"<span class='rr-b fiab {_cl}'>pertinence "
                            f"{x['pertinence']:.0f} · {_echapper(_f)}</span>")
            elif "couverture" in x and pd.notna(x.get("couverture")):
                bloc.append(f"<span class='rr-b'>{int(x['couverture'])} % des termes</span>")
            bloc.append("</div>")
            bloc.append(f"<div class='rr-benef'><span>Bénéficiaires&nbsp;:</span> "
                        f"{_echapper(str(x.get('BENEFICIAIRES') or '—'))}</div>")

            _raison = _txt(x.get("raison"))
            if _raison:
                bloc.append(f"<div class='rr-raison'>▸ {_echapper(_raison)}</div>")

            if resume:
                _ria = _txt(x.get("resume_ia"))
                txt = _ria or _txt(_trads.get(x["projectId"]))
                if txt:
                    _cls = "rr-resume ia" if _ria else "rr-resume"
                    bloc.append(f"<div class='{_cls}'>{_echapper(txt)}</div>")
            bloc.append(f"<div class='rr-jauge'><div style='width:{largeur:.0f}%'>"
                        f"</div></div></div>")
            h.append("".join(bloc))

    h.append("<div class='rr-pied'>Montants = contribution européenne aux "
             "partenaires français uniquement, distincte du budget total du "
             "consortium.</div></div>")
    display(HTML("".join(h)))
    return res


## 8 · Réponse rédigée (avec LLM, optionnel)

Le modèle ne reçoit que les projets retrouvés et **doit citer un identifiant
après chaque affirmation**. C'est la seule façon de distinguer une synthèse
fidèle d'une invention.

> Le jeton d'accès ne doit jamais être écrit en clair dans un notebook partagé.
> Sur Kaggle, utilisez **Add-ons → Secrets**.

In [34]:
SYSTEME = """Tu réponds à des questions sur des projets financés par l'Union européenne,
à partir des SEULS projets fournis dans le contexte.

Règles impératives :
- Cite l'identifiant entre crochets après chaque affirmation, ex. [101006653].
- Si l'information ne figure pas dans le contexte, dis-le explicitement.
- N'invente jamais un montant, un bénéficiaire ni une date.
- Réponds en français, de façon concise et factuelle."""

def appeler_llm(messages):
    """Appel générique. Gère le protocole Anthropic et le protocole
    compatible OpenAI (Mistral, Groq, Ollama, Albert…)."""
    import requests

    if LLM_PROTOCOLE == "anthropic":
        # Anthropic sépare le message système du fil de conversation
        systeme = next((m["content"] for m in messages if m["role"]=="system"), "")
        echanges = [m for m in messages if m["role"] != "system"]
        r = requests.post(LLM_URL,
            headers={"x-api-key": LLM_CLE,
                     "anthropic-version": "2023-06-01",
                     "content-type": "application/json"},
            json={"model": LLM_MODELE, "max_tokens": 1500,
                  "temperature": 0.2, "system": systeme, "messages": echanges},
            timeout=90)
        r.raise_for_status()
        return "".join(b.get("text","") for b in r.json().get("content", []))

    r = requests.post(LLM_URL,
        headers={"Authorization": f"Bearer {LLM_CLE}", "Content-Type": "application/json"},
        json={"model": LLM_MODELE, "messages": messages, "temperature": 0.2},
        timeout=90)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def repondre_redige(question, historique=None):
    r=recuperer(question, k=K_CONTEXTE)
    if r.empty:
        return f"Aucun projet ne correspond à « {question} »."
    contexte="\n\n".join(
        f"[{x['projectId']}] {x['title']}\n"
        f"Bénéficiaires FR : {x['BENEFICIAIRES']} — "
        f"{pd.to_numeric(x['CONTRIBUTION_FR'],errors='coerce'):,.0f} € — "
        f"{int(x['ANNEE']) if pd.notna(x['ANNEE']) else '?'}\n"
        f"{str(x.get('objective') or x.get('RAPPORT') or '')[:700]}"
        for _,x in r.iterrows())

    msgs=[{"role":"system","content":SYSTEME}]
    if historique: msgs += historique[-4:]
    msgs.append({"role":"user","content":f"CONTEXTE :\n{contexte}\n\nQUESTION : {question}"})
    try:
        return appeler_llm(msgs)
    except Exception as e:
        return (f"⚠️ Modèle injoignable ({type(e).__name__}). Réponse factuelle :\n\n"
                + repondre_factuel(question))

def repondre(question, historique=None):
    return repondre_redige(question, historique) if (LLM_ACTIF and LLM_CLE) \
           else repondre_factuel(question)


## 9 · Interface de conversation

Posez vos questions dans le champ ci-dessous. L'historique est conservé pendant
la session, ce qui permet les questions de suite (« et en 2023 ? »).

In [35]:
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
    _WIDGETS = True
except ImportError:
    _WIDGETS = False
    print("ipywidgets indisponible — utilisez discuter() en section 7.")

if _WIDGETS:
    historique = []
    saisie = widgets.Text(placeholder="Votre question… ex. : projets sur la voiture à hydrogène",
                          layout=widgets.Layout(width="80%"))
    bouton = widgets.Button(description="Demander", button_style="primary")
    efface = widgets.Button(description="Effacer")
    sortie = widgets.Output()

    def _poser(_=None):
        q = saisie.value.strip()
        if not q: return
        with sortie:
            if LLM_ACTIF and LLM_CLE:
                rep = repondre_redige(q, historique)
                display(Markdown(f"---\n**❓ {q}**"))
                display(Markdown(rep))
                afficher_resultats(q)
            else:
                rep = repondre_factuel(q)
                afficher_resultats(q)
        historique.append({"role": "user", "content": q})
        historique.append({"role": "assistant", "content": rep[:1500]})
        saisie.value = ""

    def _vider(_=None):
        historique.clear()
        with sortie: clear_output()

    bouton.on_click(_poser); saisie.on_submit(_poser); efface.on_click(_vider)
    mode = "rédigé (LLM)" if (LLM_ACTIF and LLM_CLE) else "factuel (sans LLM)"
    display(Markdown(f"**Mode : {mode}** — {len(df)} projets · moteur {MOTEUR}"))
    display(widgets.HBox([saisie, bouton, efface]), sortie)


/tmp/ipykernel_58/3197798064.py:37: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  bouton.on_click(_poser); saisie.on_submit(_poser); efface.on_click(_vider)


**Mode : rédigé (LLM)** — 77281 projets · moteur BM25

Output()

## 10 · En ligne de commande

Si les widgets ne s'affichent pas, cette boucle fonctionne partout.

In [36]:
def discuter():
    print("Posez vos questions ('quitter' pour sortir).\n")
    hist=[]
    while True:
        q=input("❓ ").strip()
        if q.lower() in ("quitter","exit","q",""): break
        rep=repondre(q, hist)
        print("\n"+rep+"\n"+"-"*70)
        hist += [{"role":"user","content":q},
                 {"role":"assistant","content":rep[:1500]}]

# discuter()


## Limites à connaître

- **Le système ne connaît que ce qu'il a retrouvé.** Si la recherche rate les
  bons projets, la réponse sera fausse même bien rédigée. La qualité de la
  recherche prime sur celle du modèle.
- **Périmètre** : projets en gestion directe à participation française. Rien sur
  la gestion partagée (Kohesio), ni sur les autres pays.
- **Les montants affichés** sont la contribution européenne aux **partenaires
  français**, distincte du budget total du consortium.
- **Le corpus est en anglais**, les questions en français : le lexique de la
  section 3 fait le pont, il est à compléter selon vos thématiques.
- **Sans jeu d'évaluation**, aucune garantie : un moteur renvoie toujours une
  liste, rien n'indique à l'œil si des projets pertinents manquent.
